# 03 — Selecting & Filtering
**Goal:** Get exactly the rows and columns you need — the most frequent operation in any analysis.

Topics:
- `loc` — select by label
- `iloc` — select by position
- Boolean indexing — filter by condition
- `query()` — SQL-like syntax
- `isin()`, `between()`, `str.contains()`

In [12]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/funnel_data.csv', parse_dates=['date'])

# Add CVR column for examples
df['cvr'] = df['activacion_tarjeta'] / df['visita_landing'] * 100

print(df.shape)
df.head(3)

(450, 13)


,date,channel,visita_landing,inicio_solicitud,datos_personales,otp,datos_financieros,carga_documentos,evaluacion_crediticia,aprobacion,firma_digital,activacion_tarjeta,cvr
0,2024-01-01,organic,1010,388,291,207,165,99,89,48,39,27,2.673267
1,2024-01-01,paid,1021,321,240,163,130,78,70,32,26,18,1.762977
2,2024-01-01,email,481,202,151,102,89,53,47,25,20,14,2.910603


## 1. `loc` — select by label (row label, column name)
`df.loc[row_selector, column_selector]`

In [13]:
# Select specific columns by name
df.loc[:, ['date', 'channel', 'cvr']].head(5)   # : = all rows

,date,channel,cvr
0,2024-01-01,organic,2.673267
1,2024-01-01,paid,1.762977
2,2024-01-01,email,2.910603
3,2024-01-01,social,1.351351
4,2024-01-01,direct,2.325581


In [14]:
# Select specific rows by index label
df.loc[0:3, ['date', 'channel', 'cvr']]   # rows 0 to 3 (inclusive)

,date,channel,cvr
0,2024-01-01,organic,2.673267
1,2024-01-01,paid,1.762977
2,2024-01-01,email,2.910603
3,2024-01-01,social,1.351351


In [15]:
# loc with boolean condition as row selector
# Get only organic channel rows
df.loc[df['channel'] == 'organic', ['date', 'cvr', 'activacion_tarjeta']].head(5)

,date,cvr,activacion_tarjeta
0,2024-01-01,2.673267,27
5,2024-01-02,2.651113,25
10,2024-01-03,2.660754,24
15,2024-01-04,2.615063,25
20,2024-01-05,2.650823,29


## 2. `iloc` — select by position (integer index)
`df.iloc[row_int, col_int]` — use when you care about position, not labels

In [16]:
# First 3 rows, first 4 columns
df.iloc[:3, :4]

,date,channel,visita_landing,inicio_solicitud
0,2024-01-01,organic,1010,388
1,2024-01-01,paid,1021,321
2,2024-01-01,email,481,202


In [17]:
# Last row
df.iloc[-1]

# Every other row (useful for sampling)
# df.iloc[::2]

date                     2024-03-30 00:00:00
channel                               direct
visita_landing                           213
inicio_solicitud                          85
datos_personales                          63
otp                                       42
datos_financieros                         33
carga_documentos                          19
evaluacion_crediticia                     17
aprobacion                                 9
firma_digital                              7
activacion_tarjeta                         4
cvr                                 1.877934
Name: 449, dtype: object

## 3. Boolean indexing — filter by condition
The most common way to filter in practice.

In [18]:
# Single condition
paid = df[df['channel'] == 'paid']
print(f'Paid rows: {len(paid)}')

# Multiple conditions — use & (and) | (or), always with parentheses
high_cvr_paid = df[(df['channel'] == 'paid') & (df['cvr'] > 2.0)]
print(f'Paid rows with CVR > 2%: {len(high_cvr_paid)}')

# OR condition
top_channels = df[(df['channel'] == 'email') | (df['channel'] == 'organic')]
print(f'Email or organic rows: {len(top_channels)}')

Paid rows: 90
Paid rows with CVR > 2%: 0
Email or organic rows: 180


In [19]:
# Negate a condition with ~
not_social = df[~(df['channel'] == 'social')]
print(f'All channels except social: {len(not_social)}')
print(not_social['channel'].unique())

All channels except social: 360
<ArrowStringArray>
['organic', 'paid', 'email', 'direct']
Length: 4, dtype: str


## 4. `isin()` — filter by a list of values

In [20]:
# Much cleaner than multiple OR conditions
top_channels = df[df['channel'].isin(['email', 'organic', 'direct'])]
print(f'Top 3 channels: {len(top_channels)} rows')
print(top_channels['channel'].value_counts())

Top 3 channels: 270 rows
channel
organic    90
email      90
direct     90
Name: count, dtype: int64


## 5. `between()` — filter numeric ranges

In [21]:
# Days where CVR was between 2% and 3%
mid_cvr = df[df['cvr'].between(2.0, 3.0)]
print(f'Days with CVR 2-3%: {len(mid_cvr)}')

# Date range filter
jan = df[df['date'].between('2024-01-01', '2024-01-31')]
print(f'January rows: {len(jan)}')

Days with CVR 2-3%: 261
January rows: 155


## 6. `str.contains()` — filter by string pattern

In [22]:
# Useful when channel names follow a pattern (e.g. paid_search, paid_social)
# Simulate a campaign column
df_paid = df[df['channel'] == 'paid'].copy()
campaigns = ['brand_search', 'non_brand_search', 'remarketing', 'brand_search', 'display']
df_paid['campaign'] = (campaigns * (len(df_paid) // 5 + 1))[:len(df_paid)]

# Filter campaigns containing 'brand'
brand = df_paid[df_paid['campaign'].str.contains('brand', case=False)]
print('Brand campaigns:', brand['campaign'].value_counts().to_dict())

Brand campaigns: {'brand_search': 36, 'non_brand_search': 18}


## 7. `query()` — SQL-like syntax
Cleaner to read for complex conditions, especially useful coming from SQL background.

In [23]:
# query() reads like SQL WHERE clause
result = df.query("channel == 'paid' and cvr > 2.0")
print(len(result))

# Use @ to reference a Python variable inside query
threshold = 2.5
channels  = ['email', 'organic']
result2   = df.query("channel in @channels and cvr > @threshold")
print(len(result2))

# Date filtering in query
result3 = df.query("date >= '2024-02-01' and channel == 'organic'")
print(len(result3))

0
180
59


## 8. `nlargest` / `nsmallest` — top N rows

In [24]:
# Top 5 days by CVR
df.nlargest(5, 'cvr')[['date', 'channel', 'cvr', 'activacion_tarjeta']]

,date,channel,cvr,activacion_tarjeta
367,2024-03-14,email,3.071672,18
437,2024-03-28,email,3.061224,18
427,2024-03-26,email,3.015075,18
297,2024-02-29,email,3.010753,14
112,2024-01-23,email,3.007519,16


In [25]:
# Bottom 5 days by activations
df.nsmallest(5, 'activacion_tarjeta')[['date', 'channel', 'activacion_tarjeta', 'cvr']]

,date,channel,activacion_tarjeta,cvr
68,2024-01-14,social,3,1.094891
98,2024-01-20,social,3,1.098901
13,2024-01-03,social,4,1.212121
18,2024-01-04,social,4,1.126761
23,2024-01-05,social,4,1.149425


## Summary
| Tool | Use when |
|---|---|
| `df['col']` | Select a single column |
| `df[['a','b']]` | Select multiple columns |
| `df.loc[rows, cols]` | Select by label |
| `df.iloc[rows, cols]` | Select by position |
| `df[df['col'] == val]` | Filter by condition |
| `&` / `\|` / `~` | AND / OR / NOT for multiple conditions |
| `df['col'].isin([...])` | Filter by list of values |
| `df['col'].between(a, b)` | Filter numeric or date range |
| `df['col'].str.contains('x')` | Filter by string pattern |
| `df.query("...")` | SQL-style filtering |
| `df.nlargest(N, 'col')` | Top N rows by a column |

**Next:** `04_cleaning.ipynb` — nulls, duplicates, type casting, rename.